# 01 · Schema Overview

**Week 1 Goal**: load every raw dataset, print shape / dtypes / null counts / sample rows, and capture findings into `docs/data_dictionary.md`.

Do **not** commit notebook outputs (handled by `nbstripout` in pre-commit, or strip manually before commit).

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

# Resolve repo root so this notebook works regardless of cwd.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'DataSet').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
DATA_DIR = REPO_ROOT / 'DataSet'
assert DATA_DIR.exists(), f'DataSet folder not found from {Path.cwd()}'
print('Repo root:', REPO_ROOT)
print('Data dir :', DATA_DIR)
sorted(p.name for p in DATA_DIR.iterdir() if p.is_file())

Repo root: /Users/uttkarshnarayan/projects/northeastern_visualization
Data dir : /Users/uttkarshnarayan/projects/northeastern_visualization/DataSet


['ReadMe.md',
 'faculty-list-2025.xlsx',
 'grants-with-abstract.xlsx',
 'grants-with-coPI.xlsx',
 'ri_matches_grants_2026.xlsx']

In [2]:
FILES = {
    'faculty':       'faculty-list-2025.xlsx',
    'grants_abs':    'grants-with-abstract.xlsx',
    'grants_copi':   'grants-with-coPI.xlsx',
    'ri_matches':    'ri_matches_grants_2026.xlsx',
}

def load_all_sheets(path: Path) -> dict[str, pd.DataFrame]:
    """Load every sheet from an xlsx into a dict[sheet_name -> DataFrame]."""
    xl = pd.ExcelFile(path)
    return {name: xl.parse(name) for name in xl.sheet_names}

raw: dict[str, dict[str, pd.DataFrame]] = {
    key: load_all_sheets(DATA_DIR / fname) for key, fname in FILES.items()
}

# Summary of sheets per file
for key, sheets in raw.items():
    print(f'{key:12s} → {FILES[key]}')
    for sheet_name, df in sheets.items():
        print(f'    sheet={sheet_name!r:35s} shape={df.shape}')
    print()

faculty      → faculty-list-2025.xlsx
    sheet='Sheet1'                            shape=(2232, 9)

grants_abs   → grants-with-abstract.xlsx
    sheet='Grants with abstract (2).csv'      shape=(8075, 25)

grants_copi  → grants-with-coPI.xlsx
    sheet='Grants with co PI indicator (1)'   shape=(3136, 22)

ri_matches   → ri_matches_grants_2026.xlsx
    sheet='ri_matches_grants_2026-2-1_3-50'   shape=(3146, 22)



## Per-dataset profile

For each dataset we print:
1. Shape
2. Column → dtype
3. Null count per column
4. First 5 rows
5. `describe(include='all')` summary

If a file has multiple sheets we profile only the first; revisit in Week 2 if other sheets matter.

In [3]:
def profile(df: pd.DataFrame, label: str) -> None:
    print('=' * 80)
    print(f'{label}  —  shape={df.shape}')
    print('=' * 80)
    print('\n--- dtypes ---')
    print(df.dtypes.to_string())
    print('\n--- null counts (non-zero only) ---')
    nulls = df.isna().sum()
    print(nulls[nulls > 0].sort_values(ascending=False).to_string() or '(none)')
    print('\n--- head(5) ---')
    display(df.head(5))
    print('\n--- describe(include="all") ---')
    display(df.describe(include='all').T)

def first_sheet(sheets: dict[str, pd.DataFrame]) -> pd.DataFrame:
    return next(iter(sheets.values()))

### Faculty list

In [4]:
profile(first_sheet(raw['faculty']), 'faculty-list-2025.xlsx')

faculty-list-2025.xlsx  —  shape=(2232, 9)

--- dtypes ---
Employee ID                      int64
Superior_Academic_Unit          object
Superior_Academic_Unit_Code     object
Academic Unit                   object
Academic Track Type             object
Academic Rank                   object
Tenure Status                   object
Location_Address_Country        object
black                          float64

--- null counts (non-zero only) ---
black            2232
Tenure Status    1274

--- head(5) ---


,Employee ID,Superior_Academic_Unit,Superior_Academic_Unit_Code,Academic Unit,Academic Track Type,Academic Rank,Tenure Status,Location_Address_Country,black
0,26501,College of Professional Studies,CC067,College of Professional Studies,Non-Tenure,Teaching Professor,NaN,United States of America,NaN
1,27481,College of Professional Studies,CC067,College of Professional Studies,Non-Tenure,Professor of the Practice,NaN,United States of America,NaN
2,27675,College of Engineering,CC044,Mechanical and Industrial Engineering,Non-Tenure,Associate Teaching Professor,NaN,United States of America,NaN
3,28441,Bouvé College of Health Sciences,CC004,Health Sciences,Non-Tenure,Assistant Teaching Professor,NaN,United States of America,NaN
4,30868,College of Professional Studies,CC067,College of Professional Studies,Non-Tenure,Associate Teaching Professor,NaN,United States of America,NaN



--- describe(include="all") ---


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Employee ID,2232.0,NaN,NaN,NaN,1584340.617384,954504.597263,26501.0,933378.5,1672080.5,2477151.5,3189672.0
Superior_Academic_Unit,2232,12,College of Engineering,356,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Superior_Academic_Unit_Code,2232,12,CC044,356,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Academic Unit,2232,80,Khoury College of Computer Sciences,210,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Academic Track Type,2232,7,Non-Tenure,1079,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Academic Rank,2232,35,Professor,433,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Tenure Status,958,3,Tenured,646,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Location_Address_Country,2232,3,United States of America,2054,NaN,NaN,NaN,NaN,NaN,NaN,NaN
black,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Grants with abstract

In [5]:
profile(first_sheet(raw['grants_abs']), 'grants-with-abstract.xlsx')

grants-with-abstract.xlsx  —  shape=(8075, 25)

--- dtypes ---
Id                                                         int64
PersonId                                                   int64
Sourcetype                                                object
SourceActivityId                                          object
DesiredVisibility                                          int64
CreatedDate                                       datetime64[ns]
UpdatedDate                                       datetime64[ns]
DeprecatedDate                                           float64
Start Date                                        datetime64[ns]
End Date                                          datetime64[ns]
Ongoing                                                  float64
Title                                                     object
Sponsor                                                  float64
Dollar Amount                                              int64
Funding Status             

,Id,PersonId,Sourcetype,SourceActivityId,DesiredVisibility,CreatedDate,UpdatedDate,DeprecatedDate,Start Date,End Date,Ongoing,Title,Sponsor,Dollar Amount,Funding Status,Proposal/Award/Contract ID,University Grant ID,URL/Link,Abstract,AACSB - Type of Intellectual Contribution,AACSB - Mission,AACSB - Portfolio of Intellectual Contribution,Type of Funding,Funding Source,Community-engaged activity?
0,91740,15186,AA,131902,2,2019-04-12 06:02:00,NaT,NaN,2005-07-01,2009-06-30,NaN,Some determinants of speech perception,NaN,941956,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,91741,15486,AA,1169275,2,2019-04-12 06:02:00,NaT,NaN,2016-09-15,2019-08-31,NaN,TWC: Small: Collaborative: An Iterative Approa...,NaN,265954,1.0,NaN,NaN,NaN,Secure multi-party computation (MPC) allows se...,NaN,NaN,NaN,NaN,NaN,NaN
2,91708,15536,AA,1120636,2,2019-04-12 06:02:00,NaT,NaN,2016-08-01,2019-07-31,NaN,CHS: Small: Collaborative Research: Teleoperat...,NaN,180000,1.0,NaN,NaN,NaN,Magnetic resonance imaging (MRI) is a widely u...,NaN,NaN,NaN,NaN,NaN,NaN
3,91756,15766,AA,366371,2,2019-04-12 06:02:00,NaT,NaN,2004-02-01,2010-01-31,NaN,"Track 2, GK-12: Northeastern University GK-12 ...",NaN,1320241,1.0,NaN,NaN,NaN,PROJECT SUMMARY andlt.br/andgt.Title of Projec...,NaN,NaN,NaN,NaN,NaN,NaN
4,91709,15604,AA,734740,2,2019-04-12 06:02:00,NaT,NaN,2010-08-16,2013-07-31,NaN,TEST OF ACCURATE PERCEPTION OF PATIENTS' AFFEC...,NaN,342594,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



--- describe(include="all") ---


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Id,8075.0,NaN,NaN,NaN,107841.622291,91518.0,93844.5,112559.0,115288.5,158405.0,14403.568773
PersonId,8075.0,NaN,NaN,NaN,131947.511331,1183.0,15381.0,15718.0,130544.0,902666.0,230292.36943
Sourcetype,8075,2,Institution,4492,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SourceActivityId,8074.0,7475.0,1320968.0,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
DesiredVisibility,8075.0,NaN,NaN,NaN,1.382539,0.0,0.0,2.0,2.0,2.0,0.923997
CreatedDate,8075,NaN,NaN,NaN,2023-02-16 20:19:58.016558080,2018-02-27 23:31:00,2019-08-02 20:34:00,2025-02-12 15:24:02.640000,2025-02-14 10:29:28.877000192,2025-11-30 01:26:55.800000,NaN
UpdatedDate,3072,NaN,NaN,NaN,2024-08-13 15:04:25.656876800,2019-11-27 07:51:00,2025-01-02 18:35:00,2025-03-20 01:22:25.193499904,2025-03-20 01:26:18.128000,2026-01-10 01:47:03.300000,NaN
DeprecatedDate,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Start Date,8075,NaN,NaN,NaN,2017-12-24 20:22:37.077399296,1995-06-01 00:00:00,2012-09-20 12:00:00,2019-09-15 00:00:00,2023-10-01 00:00:00,2026-01-01 00:00:00,NaN
End Date,8073,NaN,NaN,NaN,2021-03-19 16:02:29.832776192,1998-05-31 00:00:00,2016-02-29 00:00:00,2023-04-30 00:00:00,2026-06-30 00:00:00,2031-09-30 00:00:00,NaN


### Grants with co-PI

In [6]:
profile(first_sheet(raw['grants_copi']), 'grants-with-coPI.xlsx')

grants-with-coPI.xlsx  —  shape=(3136, 22)

--- dtypes ---
GrantId                     int64
AgencyCode                 object
AgencyGrantId              object
GrantName                  object
DurationInYears           float64
AwardDate          datetime64[ns]
DollarsPerYear              int64
StartDate          datetime64[ns]
EndDate            datetime64[ns]
PersonId                    int64
ClientFacultyId             int64
PersonName                 object
OrcidId                    object
InstitutionName            object
PI                         object
TotalDollars                int64
IsResearch                  int64
AgencyName                 object
CountryCode                object
IsGovernment                int64
IsCoPI                      int64
StartDateYear               int64

--- null counts (non-zero only) ---
AwardDate        3087
OrcidId           329
AgencyGrantId      86

--- head(5) ---


,GrantId,AgencyCode,AgencyGrantId,GrantName,DurationInYears,AwardDate,DollarsPerYear,StartDate,EndDate,PersonId,ClientFacultyId,PersonName,OrcidId,InstitutionName,PI,TotalDollars,IsResearch,AgencyName,CountryCode,IsGovernment,IsCoPI,StartDateYear
0,38584,DOED,P200A040251,Graduate Assistance in Areas of National Need,3.3333,2006-07-20,150000,2004-04-12,2007-08-14,272245,141701,"WADIA-FASCETTI, SARA J",NaN,Northeastern University,SARA J WADIA-FASCETTI,500000,1,Department Of Education,US,1,0,2004
1,42832,DOED,H133G070150,National Institute on Disability and Rehabilit...,3.0833,2008-08-15,192753,2009-08-07,2012-09-30,271613,141778,"SCHLOSSER, RALF W",0000-0002-2069-3911,Northeastern University,RALF W SCHLOSSER,594324,1,Department Of Education,US,1,0,2009
2,43588,DOED,P016A070059,Undergraduate International Studies and Foreig...,2.0000,2008-06-05,87419,2008-06-05,2010-06-30,88237,141136,"SULLIVAN, DENIS J",NaN,Northeastern University,DENIS SULLIVAN,174838,1,Department Of Education,US,1,0,2008
3,59320,DOE,FG02-08ER46487,FOCUSED SESSIONS IN 2008 SANIBEL SYMPOSIUM ON ...,1.0000,NaT,10000,2008-03-18,2009-01-14,210401,2878915,"CHENG, HAI-PING",0000-0001-5990-1725,Northeastern University,HAI-PING CHENG,10000,1,Department Of Energy,US,1,0,2008
4,63435,NASA,NNX06AF13G,UNDERSTANDING HYDROLOGIC SCALING IN VARIED LAN...,4.1666,2008-04-30,148106,2006-09-05,2010-11-26,192252,1906270,"BEIGHLEY, EDWARD EDWARD",0000-0003-1139-6226,Northeastern University,"beighley, edward",617111,1,National Aeronautics and Space Administration,US,1,0,2006



--- describe(include="all") ---


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
GrantId,3136.0,NaN,NaN,NaN,1141595.595344,38584.0,769422.0,1219671.0,1547494.0,1795307.0,478996.578558
AgencyCode,3136,21,NSF,2123,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AgencyGrantId,3050.0,2572.0,1638302.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
GrantName,3136,2459,Grant,28,NaN,NaN,NaN,NaN,NaN,NaN,NaN
DurationInYears,3136.0,NaN,NaN,NaN,3.451495,1.0,1.916666,3.3333,4.75,20.0833,1.797059
AwardDate,49,NaN,NaN,NaN,1928-10-24 22:31:50.204081664,1900-01-02 00:00:00,1900-01-02 00:00:00,1900-01-02 00:00:00,2006-04-30 00:00:00,2012-09-14 00:00:00,NaN
DollarsPerYear,3136.0,NaN,NaN,NaN,211982.080676,527.0,69971.5,126357.0,235615.75,3625548.0,309347.698745
StartDate,3136,NaN,NaN,NaN,2015-10-27 20:15:27.551020288,1995-06-01 00:00:00,2010-09-01 00:00:00,2016-09-01 00:00:00,2020-10-01 00:00:00,2026-01-01 00:00:00,NaN
EndDate,3136,NaN,NaN,NaN,2019-05-02 16:31:50.204081664,1998-05-31 00:00:00,2014-07-31 00:00:00,2020-05-31 00:00:00,2024-08-31 00:00:00,2030-09-30 00:00:00,NaN
PersonId,3136.0,NaN,NaN,NaN,447074.486288,855.0,121483.0,355521.0,743184.0,2139205.0,393404.370323


### Research-interest matches

In [7]:
profile(first_sheet(raw['ri_matches']), 'ri_matches_grants_2026.xlsx')

ri_matches_grants_2026.xlsx  —  shape=(3146, 22)

--- dtypes ---
grantid                     int64
agencycode                 object
agencygrantid              object
grantname                  object
durationinyears           float64
awarddate          datetime64[ns]
dollarsperyear              int64
startdate          datetime64[ns]
enddate            datetime64[ns]
AAUID                       int64
clientfacultyid             int64
orcid                      object
personname                 object
institutionname            object
pi                         object
totaldollars                int64
isresearch                  int64
agencyname                 object
countrycode                object
isgovernment                int64
iscopi                      int64
startdateyear               int64

--- null counts (non-zero only) ---
awarddate        3097
orcid             330
agencygrantid      86

--- head(5) ---


,grantid,agencycode,agencygrantid,grantname,durationinyears,awarddate,dollarsperyear,startdate,enddate,AAUID,clientfacultyid,orcid,personname,institutionname,pi,totaldollars,isresearch,agencyname,countrycode,isgovernment,iscopi,startdateyear
0,1171240,NSF,9501172,CAREER: Architectural Support for Object-orie...,2.9166,NaT,45255,1995-06-01,1998-05-31,149515,148296,0000-0002-5692-0151,"KAELI, DAVID R",Northeastern University,"KAELI, DAVID",131995,1,National Science Foundation,US,1,0,1995
1,1171644,NSF,9702257,CAREER: Connecting the Engineering Profession ...,5.4166,NaT,66255,1997-07-01,2002-12-31,60253,2983576,0000-0002-6719-6023,"LESKO, JACK JAMES",Northeastern University,"LESKO, JOHN",358885,1,National Science Foundation,US,1,0,1997
2,1171799,NSF,9701998,CAREER: Driven Interfaces in Random Media,4.9166,NaT,45294,1997-06-01,2002-05-31,6001,404727,0000-0002-4028-3522,"BARABASI, ALBERT-LASZLO",Northeastern University,"BARABASI, ALBERT-LASZLO",222700,1,National Science Foundation,US,1,0,1997
3,1172215,NSF,9703384,CAREER: Investigating Research Issues in Ubiqu...,2.1666,NaT,67409,1997-07-01,2002-06-30,105382,2197563,0000-0002-3408-587X,"ABOWD, GREGORY DOMINIC",Northeastern University,"ABOWD, GREGORY",388376,1,National Science Foundation,US,1,0,1997
4,1172687,NSF,9702656,CAREER: Predicting Bridge Life-Cycle Deteriora...,6.9166,NaT,48289,1997-07-01,2004-06-30,272245,141701,NaN,"WADIA-FASCETTI, SARA J",Northeastern University,"WADIA-FASCETTI, SARA",334000,1,National Science Foundation,US,1,0,1997



--- describe(include="all") ---


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
grantid,3146.0,NaN,NaN,NaN,1143167.720915,38584.0,791135.25,1224937.5,1548396.0,1797758.0,479345.619983
agencycode,3146,21,NSF,2131,NaN,NaN,NaN,NaN,NaN,NaN,NaN
agencygrantid,3060.0,2578.0,619616.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
grantname,3146,2468,Grant,28,NaN,NaN,NaN,NaN,NaN,NaN,NaN
durationinyears,3146.0,NaN,NaN,NaN,3.532016,1.0,2.0833,3.6666,4.9166,21.0833,1.834592
awarddate,49,NaN,NaN,NaN,1928-10-24 22:31:50.204081664,1900-01-02 00:00:00,1900-01-02 00:00:00,1900-01-02 00:00:00,2006-04-30 00:00:00,2012-09-14 00:00:00,NaN
dollarsperyear,3146.0,NaN,NaN,NaN,206606.774952,527.0,69853.75,124962.5,229343.0,3625548.0,301805.141385
startdate,3146,NaN,NaN,NaN,2015-11-05 07:09:48.175460864,1995-06-01 00:00:00,2010-09-01 00:00:00,2016-09-01 00:00:00,2020-10-01 00:00:00,2026-01-01 00:00:00,NaN
enddate,3146,NaN,NaN,NaN,2019-06-10 03:56:11.137953024,1998-05-31 00:00:00,2014-07-31 00:00:00,2020-06-30 00:00:00,2024-09-30 00:00:00,2030-09-30 00:00:00,NaN
AAUID,3146.0,NaN,NaN,NaN,448625.938652,855.0,121483.0,355521.0,743184.0,2139205.0,395765.539426


## Cross-file column overlap (candidate join keys)

Quick check: which column names (case-insensitive) appear in more than one file? These are the first candidates for join keys; confirm semantics in Week 2.

In [8]:
from collections import defaultdict

col_to_files: dict[str, list[str]] = defaultdict(list)
for key, sheets in raw.items():
    df = first_sheet(sheets)
    for col in df.columns:
        col_to_files[str(col).strip().lower()].append(key)

shared = {c: files for c, files in col_to_files.items() if len(files) > 1}
pd.DataFrame(
    [(c, ', '.join(sorted(set(files)))) for c, files in sorted(shared.items())],
    columns=['column (lower)', 'appears in'],
)

,column (lower),appears in
0,agencycode,"grants_copi, ri_matches"
1,agencygrantid,"grants_copi, ri_matches"
2,agencyname,"grants_copi, ri_matches"
3,awarddate,"grants_copi, ri_matches"
4,clientfacultyid,"grants_copi, ri_matches"
5,countrycode,"grants_copi, ri_matches"
6,dollarsperyear,"grants_copi, ri_matches"
7,durationinyears,"grants_copi, ri_matches"
8,enddate,"grants_copi, ri_matches"
9,grantid,"grants_copi, ri_matches"


## Action items → `docs/data_dictionary.md`

After running this notebook, copy the column lists into the data dictionary tables and annotate:
- Inferred type (after coercion)
- Nullable Y/N
- Units, value ranges, normalization rules, candidate join keys

### Notes already resolved (no advisor input needed)
- **`black` column in faculty list** — leftover empty field for some reason. Drop entirely.
- **`grants-with-abstract` role** — this file is the **text-content companion** to the structured grants tables. Its differentiator is `Title` + `Abstract`. The 100%-null structured columns (`Sponsor`, `Funding Source`, `Type of Funding`, `AACSB-*`, etc.) are expected — that information lives in `grants-with-coPI` / `ri_matches_grants_2026`. Use `grants-with-abstract` as a join-on-grant-id enrichment for NLP / topic analysis in Week 8, not as a standalone fact table.

### Open questions for the Week 1 advisor sync

These require institutional / stewardship context — I can't settle them from the data alone.

1. **Faculty identity crosswalk.** Three identifier schemes appear and don't obviously line up:
   - `faculty-list-2025` → `Employee ID` (5-digit, e.g., 26501)
   - `grants-with-abstract` → `PersonId` (5-digit, e.g., 15186)
   - `grants-with-coPI` → `PersonId` (6-digit, e.g., 272245) **plus** `ClientFacultyId` (6-digit, e.g., 141701)
   - `ri_matches_grants_2026` → `AAUID` + `clientfacultyid` + `orcid` + `personname`
   Is there an official crosswalk? Which ID is canonical for joining grants back to the 2025 faculty roster? (Without this, a meaningful chunk of grants may end up unattributable to a current faculty member.)

2. **`grants-with-coPI` (3,136 rows) vs `ri_matches_grants_2026` (3,146 rows).** These look like near-duplicates — same columns, only snake-cased, near-identical row counts, but `ri_matches` includes pre-2005 records (e.g., 1995) that `coPI` does not. Is `ri_matches` the authoritative replacement (a superset)? Use one and drop the other, or do they cover meaningfully different scopes?

3. **Grant-ID linkage between abstract and structured files.** `grants-with-abstract.Id` (e.g., 91740) does not visibly match `grants-with-coPI.GrantId` (e.g., 38584) or `ri_matches.grantid` (e.g., 1171240) ranges. To use abstracts as a text enrichment (per the note above), I need to know how to join them. Is there an external crosswalk, or a hidden field like `SourceActivityId`?

4. **timeline scope definition.** `ri_matches` contains grants starting as early as **1995**, but there are only a few grants listed with pre **2000** listings. Is the analytical window calendar-year 2005–2025, or all available history?

> Items not on this list (currency, calendar vs fiscal year, inflation deflator, `Tenure Status` nullability) are empirically resolvable and I'll handle them in Week 2 without blocking on advisor input.
